In [36]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.preprocessing import QuantileTransformer, RobustScaler
# from sklearn.feature_selection import SelectFromModel
# from sklearn.metrics import mean_squared_error, r2_score
# from datetime import datetime
# import matplotlib.pyplot as plt
# import seaborn as sns

# print("Loading data...")
# data = pd.read_csv('./datasets/merged_data(1).csv', usecols=['state', 'discovery_month', 'discovery_doy', 't_min', 't_max', 'elevation', 'fire_size'])

# # Feature engineering - extract more information from our existing variables. 
# print("Preparing features...")
# data['sin_day'] = np.sin(2 * np.pi * data['discovery_doy']/365)
# data['cos_day'] = np.cos(2 * np.pi * data['discovery_doy']/365)
# data['temp_range'] = data['t_max'] - data['t_min']
# data['temp_squared'] = data['t_max'] ** 2
# data['temp_exp'] = np.exp((data['t_max'] - data['t_max'].mean()) / data['t_max'].std())
# data['temp_elev_interact'] = data['t_max'] * data['elevation']
# print('Features prepared... [sin_day, cos_day, temp_range, temp_squared, temp_exp, temp_elev_interact]')


# # Aggregate the data
# data_state = data.groupby(['state', 'discovery_month', 'discovery_doy']).agg({
#     'fire_size': 'sum',
#     't_min': 'mean',
#     't_max': 'mean',
#     'temp_range': 'mean',
#     'temp_squared': 'mean',
#     'temp_exp': 'mean',
#     'temp_elev_interact': 'mean',
#     'elevation': 'mean',
#     'sin_day': 'first',
#     'cos_day': 'first'
# }).reset_index()


# # Normalize fire_size to a 0-1 scale for risk_score using QuantileTransformer
# qt = QuantileTransformer(output_distribution='uniform', n_quantiles=1000)
# data_state['risk_score'] = qt.fit_transform(data_state[['fire_size']])

# # Define features and target
# features = ['state', 'discovery_month', 'discovery_doy', 't_min', 't_max', 'temp_range', 'temp_squared', 
#             'temp_exp', 'temp_elev_interact', 'elevation', 'sin_day', 'cos_day']
# target = 'risk_score'

# # Prepare the feature matrix X and target vector y
# X = pd.get_dummies(data_state[features], columns=['state'])
# y = data_state[target]

# # Split the data
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Scale features
# scaler = RobustScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Create sample weights based on temperature and fire size
# temp_transformer = QuantileTransformer(output_distribution='normal')
# temp_normalized = temp_transformer.fit_transform(X_train[['t_max']])
# fire_size_normalized = QuantileTransformer().fit_transform(data_state[data_state.index.isin(X_train.index)][['fire_size']])
# sample_weights = np.exp(temp_normalized).flatten() * np.exp(fire_size_normalized).flatten()

# # Train RandomForest model
# model = RandomForestRegressor(n_estimators=500, max_depth=20, min_samples_split=5, random_state=42, n_jobs=-1)
# model.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# # Make predictions
# y_pred = model.predict(X_test_scaled)

# # Evaluate the model
# mse = mean_squared_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)
# print(f"\nMean Squared Error: {mse}")
# print(f"R-squared Score: {r2}")

# def predict_risk(date, state, t_min, t_max):
#     # Convert date to month and day of year
#     date = pd.to_datetime(date)
#     month = date.month
#     day_of_year = date.timetuple().tm_yday
    
#     # Create a DataFrame with all features, initialized to 0
#     input_data = pd.DataFrame(0, index=[0], columns=X.columns)
    
#     # Fill in the known values
#     input_data['discovery_month'] = month
#     input_data['discovery_doy'] = day_of_year
#     input_data['t_min'] = t_min
#     input_data['t_max'] = np.clip(t_max, data['t_max'].min(), data['t_max'].max())  
#     input_data['elevation'] = data_state[data_state['state'] == state]['elevation'].mean()
#     input_data['sin_day'] = np.sin(2 * np.pi * day_of_year/365)
#     input_data['cos_day'] = np.cos(2 * np.pi * day_of_year/365)
#     input_data['temp_range'] = input_data['t_max'] - input_data['t_min']
#     input_data['temp_squared'] = input_data['t_max'] ** 2
#     input_data['temp_exp'] = np.exp((input_data['t_max'] - data['t_max'].mean()) / data['t_max'].std())
#     input_data['temp_elev_interact'] = input_data['t_max'] * input_data['elevation']
    
#     # Set the correct state column to 1
#     state_column = f'state_{state}'
#     if state_column in input_data.columns:
#         input_data[state_column] = 1
#     else:
#         print(f"Warning: State '{state}' not found in training data.")
    
#     # Scale input data
#     input_scaled = scaler.transform(input_data)
    
#     # Make prediction
#     risk_score = model.predict(input_scaled)[0]
#     return risk_score

# # Example usage of the model - REQUIRED ARGUMENTS (date, state, t_min, t_max)
# date = '2024-07-15' 
# state = 'NY'
# t_min = 26
# t_max = 40

# risk_score = predict_risk(date, state, t_min, t_max)
# print(f"\nPredicted Risk Score for {date} in {state} (T_min: {t_min}, T_max: {t_max}): {risk_score:.4f}")


In [37]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import QuantileTransformer, RobustScaler
from sklearn.metrics import mean_squared_error, r2_score
import json
from datetime import datetime, timedelta

class FireRiskPredictor:
    def __init__(self):
        self.model = None
        self.scaler = None
        self.qt = None
        self.feature_importances = None
        
    def train(self, data_path):
        print("Loading data...")
        self.data = pd.read_csv(data_path, usecols=['state', 'discovery_month', 'discovery_doy', 
                                                   't_min', 't_max', 'elevation', 'fire_size'])
        
        # Feature engineering
        self._prepare_features()
        
        # Train model
        self._train_model()
        
        # Calculate feature importance
        self._calculate_feature_importance()
    
    def _prepare_features(self):
        print("Preparing features...")
        self.data['sin_day'] = np.sin(2 * np.pi * self.data['discovery_doy']/365)
        self.data['cos_day'] = np.cos(2 * np.pi * self.data['discovery_doy']/365)
        self.data['temp_range'] = self.data['t_max'] - self.data['t_min']
        self.data['temp_squared'] = self.data['t_max'] ** 2
        self.data['temp_exp'] = np.exp((self.data['t_max'] - self.data['t_max'].mean()) / self.data['t_max'].std())
        self.data['temp_elev_interact'] = self.data['t_max'] * self.data['elevation']
        
        # Aggregate by state and month
        self.data_state = self.data.groupby(['state', 'discovery_month', 'discovery_doy']).agg({
            'fire_size': 'sum',
            't_min': 'mean',
            't_max': 'mean',
            'temp_range': 'mean',
            'temp_squared': 'mean',
            'temp_exp': 'mean',
            'temp_elev_interact': 'mean',
            'elevation': 'mean',
            'sin_day': 'first',
            'cos_day': 'first'
        }).reset_index()
        
        # Transform fire_size to risk_score
        self.qt = QuantileTransformer(output_distribution='uniform', n_quantiles=1000)
        self.data_state['risk_score'] = self.qt.fit_transform(self.data_state[['fire_size']])
    
    def _train_model(self):
        features = ['state', 'discovery_month', 'discovery_doy', 't_min', 't_max', 
                   'temp_range', 'temp_squared', 'temp_exp', 'temp_elev_interact', 
                   'elevation', 'sin_day', 'cos_day']
        
        X = pd.get_dummies(self.data_state[features], columns=['state'])
        y = self.data_state['risk_score']
        
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        self.scaler = RobustScaler()
        X_train_scaled = self.scaler.fit_transform(self.X_train)
        
        self.model = RandomForestRegressor(n_estimators=500, max_depth=20, random_state=42, n_jobs=-1)
        self.model.fit(X_train_scaled, self.y_train)
    
    def _calculate_feature_importance(self):
        feature_importance = pd.DataFrame({
            'feature': self.X_train.columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Filter out state-specific features for general importance
        self.feature_importances = feature_importance[
            ~feature_importance['feature'].str.startswith('state_')
        ].head(10)
    
    def predict_with_insights(self, date, state, t_min, t_max):
        """Predict fire risk and return additional visualization data"""
        date_obj = pd.to_datetime(date)
        risk_score = self._predict_single(date_obj, state, t_min, t_max)
        
        # Generate visualization data
        viz_data = {
            'current_prediction': {
                'risk_score': float(risk_score),
                'date': date,
                'state': state,
                'temperature': {'min': t_min, 'max': t_max}
            },
            'feature_importance': self.feature_importances.to_dict('records'),
            'historical_context': self._get_historical_context(state, date_obj),
            'temperature_risk_curve': self._generate_temp_risk_curve(state, date_obj, t_min, t_max),
            'seasonal_risk': self._calculate_seasonal_risk(state),
            'state_comparison': self._get_state_comparison(state, date_obj)
        }
        
        return viz_data
    
    def _predict_single(self, date, state, t_min, t_max):
        input_data = pd.DataFrame(0, index=[0], columns=self.X_train.columns)
        
        # Fill basic features
        input_data['discovery_month'] = date.month
        input_data['discovery_doy'] = date.timetuple().tm_yday
        input_data['t_min'] = t_min
        input_data['t_max'] = t_max
        input_data['elevation'] = self.data_state[self.data_state['state'] == state]['elevation'].mean()
        
        # Calculate derived features
        input_data['sin_day'] = np.sin(2 * np.pi * input_data['discovery_doy']/365)
        input_data['cos_day'] = np.cos(2 * np.pi * input_data['discovery_doy']/365)
        input_data['temp_range'] = input_data['t_max'] - input_data['t_min']
        input_data['temp_squared'] = input_data['t_max'] ** 2
        input_data['temp_exp'] = np.exp((input_data['t_max'] - self.data['t_max'].mean()) / self.data['t_max'].std())
        input_data['temp_elev_interact'] = input_data['t_max'] * input_data['elevation']
        
        # Set state
        state_column = f'state_{state}'
        if state_column in input_data.columns:
            input_data[state_column] = 1
        
        input_scaled = self.scaler.transform(input_data)
        return self.model.predict(input_scaled)[0]
    
    def _get_historical_context(self, state, date):
        """Get historical risk scores for the same month"""
        month = date.month
        state_data = self.data_state[
            (self.data_state['state'] == state) & 
            (self.data_state['discovery_month'] == month)
        ]
        
        return {
            'avg_risk': float(state_data['risk_score'].mean()),
            'max_risk': float(state_data['risk_score'].max()),
            'min_risk': float(state_data['risk_score'].min()),
            'percentile_90': float(state_data['risk_score'].quantile(0.9))
        }
    
    def _generate_temp_risk_curve(self, state, date, t_min, t_max):
        """Generate risk scores across temperature range"""
        temp_range = np.linspace(t_min - 5, t_max + 5, 20)
        risks = []
        
        for temp in temp_range:
            risk = self._predict_single(date, state, t_min, temp)
            risks.append({'temperature': float(temp), 'risk': float(risk)})
        
        return risks
    
    def _calculate_seasonal_risk(self, state):
        """Calculate average risk scores by month for the state"""
        monthly_risks = self.data_state[self.data_state['state'] == state].groupby('discovery_month').agg({
            'risk_score': 'mean',
            't_max': 'mean',
            't_min': 'mean'
        }).reset_index()
        
        return monthly_risks.to_dict('records')
    
    def _get_state_comparison(self, target_state, date):
        """Compare risk levels across neighboring states"""
        month = date.month
        all_states = self.data_state['state'].unique()
        
        state_risks = []
        for state in all_states:
            avg_risk = self.data_state[
                (self.data_state['state'] == state) & 
                (self.data_state['discovery_month'] == month)
            ]['risk_score'].mean()
            
            state_risks.append({
                'state': state,
                'avg_risk': float(avg_risk)
            })
        
        return sorted(state_risks, key=lambda x: x['avg_risk'], reverse=True)
    
    def predict_with_visualizations(self, date, state, t_min, t_max):
        """Generate prediction and visualization data for a specific query"""
        date_obj = pd.to_datetime(date)
        base_risk = self._predict_single(date_obj, state, t_min, t_max)
        
        # 1. Temperature Sensitivity Analysis
        temp_analysis = self._analyze_temperature_sensitivity(date_obj, state, t_min, t_max)
        
        # 2. Historical Risk Comparison
        historical_comparison = self._get_historical_comparison(date_obj, state)
        
        # 3. Monthly Risk Forecast
        risk_forecast = self._generate_monthly_forecast(date_obj, state, t_min, t_max)
        
        return {
            'current_prediction': {
                'risk_score': float(base_risk),
                'date': date,
                'state': state,
                'temperature': {'min': t_min, 'max': t_max}
            },
            'temperature_sensitivity': temp_analysis,
            'historical_comparison': historical_comparison,
            'monthly_forecast': risk_forecast
        }
    
    def _analyze_temperature_sensitivity(self, date, state, t_min, t_max):
        """Analyze how risk changes with temperature variations"""
        temp_range = np.linspace(t_min - 5, t_max + 5, 15)
        base_temp = (t_max + t_min) / 2
        
        sensitivity_data = []
        for temp in temp_range:
            risk_low = self._predict_single(date, state, max(temp - 5, 0), temp)
            risk_high = self._predict_single(date, state, temp, temp + 5)
            
            sensitivity_data.append({
                'temperature': float(temp),
                'risk_low_temp': float(risk_low),
                'risk_high_temp': float(risk_high),
                'is_current': abs(temp - base_temp) < 1
            })
        
        return sensitivity_data
    
    def _get_historical_comparison(self, date, state):
        """Compare current prediction with historical data"""
        month = date.month
        day_of_year = date.timetuple().tm_yday
        
        # Get 30-day window around the target date
        historical_data = self.data_state[
            (self.data_state['state'] == state) &
            (abs(self.data_state['discovery_doy'] - day_of_year) <= 15)
        ]
        
        if len(historical_data) == 0:
            return []
        
        risks_by_temp = []
        temp_ranges = np.linspace(
            historical_data['t_max'].min(),
            historical_data['t_max'].max(),
            8
        )
        
        for i in range(len(temp_ranges)-1):
            temp_min, temp_max = temp_ranges[i], temp_ranges[i+1]
            mask = (historical_data['t_max'] >= temp_min) & (historical_data['t_max'] < temp_max)
            risk_scores = historical_data[mask]['risk_score']
            
            if len(risk_scores) > 0:
                risks_by_temp.append({
                    'temp_range': f"{temp_min:.1f}-{temp_max:.1f}°C",
                    'avg_risk': float(risk_scores.mean()),
                    'max_risk': float(risk_scores.max()),
                    'min_risk': float(risk_scores.min()),
                    'sample_size': int(len(risk_scores))
                })
        
        return risks_by_temp
    
    def _generate_monthly_forecast(self, date, state, t_min, t_max):
        """Generate risk predictions for upcoming months"""
        forecasts = []
        
        # Generate predictions for next 6 months
        for i in range(6):
            future_date = date + pd.DateOffset(months=i)
            
            # Adjust temperatures based on historical averages for that month
            month_data = self.data_state[
                (self.data_state['state'] == state) &
                (self.data_state['discovery_month'] == future_date.month)
            ]
            
            if len(month_data) > 0:
                temp_adjustment = (month_data['t_max'].mean() - t_max) / 2
                adjusted_t_max = t_max + temp_adjustment
                adjusted_t_min = t_min + temp_adjustment
                
                risk = self._predict_single(future_date, state, adjusted_t_min, adjusted_t_max)
                
                forecasts.append({
                    'month': future_date.strftime('%B'),
                    'risk_score': float(risk),
                    'avg_temp': float(adjusted_t_max),
                    'historical_avg_risk': float(month_data['risk_score'].mean())
                })
        
        return forecasts
    
if __name__ == "__main__":
    predictor = FireRiskPredictor()
    predictor.train('./datasets/merged_data(1).csv')
    
    insights = predictor.predict_with_insights(
        date='2024-07-15',
        state='NY',
        t_min=25,
        t_max=40
    )
    
    # Print results in a formatted way
    print("\nPrediction Results:")
    print(json.dumps(insights, indent=2))

Loading data...
Preparing features...

Prediction Results:
{
  "current_prediction": {
    "risk_score": 0.5484744843628289,
    "date": "2024-07-15",
    "state": "NY",
    "temperature": {
      "min": 25,
      "max": 40
    }
  },
  "feature_importance": [
    {
      "feature": "temp_elev_interact",
      "importance": 0.25091757435558987
    },
    {
      "feature": "temp_exp",
      "importance": 0.2402338328784948
    },
    {
      "feature": "temp_range",
      "importance": 0.0675955156578546
    },
    {
      "feature": "sin_day",
      "importance": 0.046443778981744184
    },
    {
      "feature": "elevation",
      "importance": 0.04605478275864443
    },
    {
      "feature": "t_min",
      "importance": 0.037186054715623876
    },
    {
      "feature": "cos_day",
      "importance": 0.035555689930263396
    },
    {
      "feature": "discovery_doy",
      "importance": 0.0285956409713
    },
    {
      "feature": "t_max",
      "importance": 0.024644904245022022
